# Rap-Snacks ColabFold Validation

Two independent sections — run them in separate sessions or sequentially:

- **Section A** — ProteinMPNN × 3 temperatures (0.1 / 0.2 / 0.3) + ColabFold self-consistency  
- **Section B** — Originals (concordance + native_ala) + Scrambled sequences → ColabFold (apples-to-apples)

**Runtime:** A100 GPU recommended.  
**Input:** Upload `rap_snacks_inputs.zip` (pack locally — instructions in cell below).  
**Output:** Results downloaded as zip at end of each section.

---
## 0 — Install & Setup
Run once per session.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'colabfold[alphafold]'], check=True)

import os
if not os.path.exists('/content/ProteinMPNN'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/dauparas/ProteinMPNN.git',
                    '/content/ProteinMPNN'], check=True)
    print('ProteinMPNN cloned')
else:
    print('ProteinMPNN already present')
print('Setup complete.')

In [ ]:
import json, random, subprocess, sys, time, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

CONTENT      = Path('/content')
MPNN_SCRIPT  = CONTENT / 'ProteinMPNN' / 'protein_mpnn_run.py'
MPNN_WEIGHTS = CONTENT / 'ProteinMPNN' / 'vanilla_model_weights'
print('Imports OK')
print(f'MPNN script: {MPNN_SCRIPT.exists()}')
print(f'MPNN weights: {MPNN_WEIGHTS.exists()}')

### Upload inputs

Pack locally from `rap-snacks-v1/` directory:
```bash
zip rap_snacks_inputs.zip \
    data/phase2_candidates.csv \
    data/phase2_fixed_positions.jsonl \
    data/aggregated_lines_v2_enriched.csv \
    outputs/boltz/boltz_summary.csv \
    data/boltz_id_map.json \
    outputs/masks/mask_v2_concordance.json \
    outputs/proteinmpnn/pdbs/*.pdb
```

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(k for k in uploaded if k.endswith('.zip'))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(CONTENT)
print(f'Extracted {zip_name}')
for p in [
    'data/phase2_candidates.csv',
    'data/phase2_fixed_positions.jsonl',
    'data/aggregated_lines_v2_enriched.csv',
    'outputs/boltz/boltz_summary.csv',
    'data/boltz_id_map.json',
    'outputs/masks/mask_v2_concordance.json',
]:
    exists = (CONTENT / p).exists()
    print(f"  [{'OK' if exists else 'MISSING'}] {p}")
pdb_count = len(list((CONTENT / 'outputs/proteinmpnn/pdbs').glob('*.pdb')))
print(f'  PDBs: {pdb_count} found')

---
## Config (shared)

In [ ]:
# MPNN
TOP_N        = 12
N_SEQS       = 50
TEMPS        = [0.1, 0.2, 0.3]   # three temperature sweeps
PLDDT_MIN    = 0.35              # ColabFold pLDDT 0-1 after /100

# Scrambled control
N_SHUFFLES   = 3
SEED         = 42

# ColabFold
NUM_RECYCLE  = 1     # 1 for filtering; use 3 for final submission seqs
MODEL_TYPE   = 'alphafold2_ptm'
NUM_MODELS   = 1

# Paths
DATA        = CONTENT / 'data'
OUTPUTS     = CONTENT / 'outputs'
PDB_DIR     = OUTPUTS / 'proteinmpnn' / 'pdbs'
DESIGNED    = OUTPUTS / 'proteinmpnn' / 'designed'
FIXED_JSONL = DATA / 'phase2_fixed_positions.jsonl'

CANDIDATES  = pd.read_csv(DATA / 'phase2_candidates.csv').head(TOP_N)
MASK_BY_BAR = {m['bar_id']: m for m in json.load(open(OUTPUTS / 'masks' / 'mask_v2_concordance.json'))}
print(f'Candidates: {len(CANDIDATES)} bars')
print(CANDIDATES[['bar_id']].to_string(index=False))

---
## Helpers (shared)

In [ ]:
def fix_boltz_chain(pdb_text):
    """Fix Boltz 3-char chain ID bug."""
    out = []
    for line in pdb_text.splitlines(keepends=True):
        if (line.startswith('ATOM') or line.startswith('HETATM')) and len(line) >= 32:
            line = line[:21] + 'A' + line[24:]
        out.append(line)
    return ''.join(out)


def parse_mpnn_fasta(fa_path):
    seqs, header, seq = [], None, []
    for line in fa_path.read_text().splitlines():
        if line.startswith('>'):
            if header: seqs.append((header, ''.join(seq)))
            header, seq = line[1:].strip(), []
        else:
            seq.append(line.strip())
    if header: seqs.append((header, ''.join(seq)))
    return seqs


def parse_colabfold_scores(scores_dir):
    """
    Parse ColabFold output dir. Returns {name: {'plddt': float(0-1), 'ptm': float}}.
    ColabFold pLDDT is 0-100 in JSON — divide by 100.
    """
    results = {}
    for p in sorted(Path(scores_dir).glob('*_scores_rank_001*.json')):
        name = p.name.split('_scores_rank_001')[0]
        data = json.loads(p.read_text())
        results[name] = {
            'plddt': float(np.mean(data['plddt'])) / 100.0,
            'ptm':   data.get('ptm', None),
        }
    return results


def run_colabfold(fasta_path, out_dir):
    """Run colabfold_batch, return parsed scores dict."""
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    cmd = [
        'colabfold_batch', str(fasta_path), str(out_dir),
        '--num-recycle', str(NUM_RECYCLE),
        '--model-type', MODEL_TYPE,
        '--num-models', str(NUM_MODELS),
    ]
    print(f'Running colabfold_batch on {fasta_path.name} ...')
    r = subprocess.run(cmd, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'ColabFold failed (exit {r.returncode})')
    return parse_colabfold_scores(out_dir)


def shuffle_sequence(seq, rng):
    chars = list(seq)
    rng.shuffle(chars)
    return ''.join(chars)


DARK_BG   = '#0e0e0e'
DARK_AX   = '#0e0e0e'
WHITE     = 'white'
GRID_COL  = '#333333'

def dark_ax(ax):
    ax.set_facecolor(DARK_AX)
    ax.tick_params(colors=WHITE)
    ax.yaxis.label.set_color(WHITE)
    ax.xaxis.label.set_color(WHITE)
    ax.title.set_color(WHITE)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_COL)

print('Helpers loaded.')

---
---
# SECTION A — ProteinMPNN × 3 Temperatures + ColabFold

Runs MPNN at temps **0.1, 0.2, 0.3** for all 12 candidate bars.
- Temp 0.1 = near-greedy (tight clustering around the most probable sequence)
- Temp 0.2 = moderate diversity
- Temp 0.3 = higher diversity — gives dropout bars (0/50 at 0.1) a chance to recover

All designs folded together in one ColabFold batch job.

In [ ]:
# A1 — Verify PDBs and build fixed_positions.jsonl
PDB_DIR.mkdir(parents=True, exist_ok=True)
DESIGNED.mkdir(parents=True, exist_ok=True)

fixed_pos_records = []
for _, row in CANDIDATES.iterrows():
    bar_id  = row['bar_id']
    pdb_dst = PDB_DIR / f'{bar_id}.pdb'
    if not pdb_dst.exists():
        print(f'  [MISSING] {bar_id}.pdb')
        continue
    pdb_dst.write_text(fix_boltz_chain(pdb_dst.read_text()))

    m = MASK_BY_BAR.get(bar_id, {})
    bjozxu = set(m.get('mutation_mask', []))
    seq_len = m.get('sequence_length', int(row['fasta_seq_len']))
    fixed_pos_records.append({
        'bar_id':  bar_id,
        'fixed':   [i+1 for i in range(seq_len) if i not in bjozxu],
        'design':  [i+1 for i in sorted(bjozxu)],
        'seq_len': seq_len,
    })
    print(f'  {bar_id}  len={seq_len}  designable={len(bjozxu)}')

# Single-object JSONL (all bars in one line — MPNN requires this)
combined = {r['bar_id']: {'A': r['fixed']} for r in fixed_pos_records}
with open(FIXED_JSONL, 'w') as f:
    f.write(json.dumps(combined) + '\n')
print(f'\nPrepared {len(fixed_pos_records)} bars.')

In [ ]:
# A2 — Run ProteinMPNN at each temperature for each bar
for temp in TEMPS:
    print(f'\n====== Temperature {temp} ======')
    for rec in fixed_pos_records:
        bar_id  = rec['bar_id']
        pdb_in  = PDB_DIR / f'{bar_id}.pdb'
        out_dir = DESIGNED / f't{temp}' / bar_id
        out_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            sys.executable, str(MPNN_SCRIPT),
            '--pdb_path',              str(pdb_in),
            '--out_folder',            str(out_dir),
            '--num_seq_per_target',    str(N_SEQS),
            '--sampling_temp',         str(temp),
            '--fixed_positions_jsonl', str(FIXED_JSONL),
            '--path_to_model_weights', str(MPNN_WEIGHTS),
            '--batch_size',            '1',
            '--suppress_print',        '1',
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'  [ERROR] {bar_id}: {r.stderr[-300:]}')
            continue
        fa = list(out_dir.rglob('*.fa')) + list(out_dir.rglob('*.fasta'))
        n = len(parse_mpnn_fasta(fa[0])) - 1 if fa else 0
        print(f'  {bar_id}  temp={temp}  → {n} designs')

print('\nMPNN complete — all temperatures.')

In [ ]:
# A3 — Collect all designs into one FASTA (all temps, all bars)
CF_MPNN_FASTA = CONTENT / 'mpnn_designs_all_temps.fasta'
design_meta   = []

with open(CF_MPNN_FASTA, 'w') as fout:
    for temp in TEMPS:
        for rec in fixed_pos_records:
            bar_id  = rec['bar_id']
            out_dir = DESIGNED / f't{temp}' / bar_id
            fa = list(out_dir.rglob('*.fa')) + list(out_dir.rglob('*.fasta'))
            if not fa: continue
            for i, (header, seq) in enumerate(parse_mpnn_fasta(fa[0])[1:]):  # skip ref
                temp_str = str(temp).replace('.', 'p')
                cf_name  = f'{bar_id}_t{temp_str}_d{i:02d}'
                fout.write(f'>{cf_name}\n{seq}\n')
                design_meta.append({
                    'cf_name': cf_name, 'bar_id': bar_id,
                    'temp': temp, 'design_idx': i,
                    'header': header, 'sequence': seq, 'seq_len': len(seq),
                })

meta_df = pd.DataFrame(design_meta)
print(f'Total designs across all temps: {len(meta_df)}')
print(meta_df.groupby(['bar_id', 'temp']).size().unstack(fill_value=0))

In [ ]:
# A4 — Run ColabFold on all designs
# ~1800 seqs (12 bars × 50 × 3 temps) on A100 ≈ 40–60 min with recycle=1
CF_MPNN_OUT = CONTENT / 'cf_mpnn_output'
scores = run_colabfold(CF_MPNN_FASTA, CF_MPNN_OUT)
print(f'Scored {len(scores)} sequences.')

In [ ]:
# A5 — Merge scores
meta_df['cf_plddt'] = meta_df['cf_name'].map(lambda n: scores.get(n, {}).get('plddt'))
meta_df['cf_ptm']   = meta_df['cf_name'].map(lambda n: scores.get(n, {}).get('ptm'))
meta_df['passes']   = meta_df['cf_plddt'].apply(lambda p: p is not None and p >= PLDDT_MIN)

MPNN_CSV = CONTENT / 'mpnn_colabfold_results.csv'
meta_df.to_csv(MPNN_CSV, index=False)
print(f'Saved {len(meta_df)} rows → {MPNN_CSV}')

# Per-bar per-temp summary
print('\n--- Pass rate by bar and temperature ---')
summary = meta_df.groupby(['bar_id', 'temp']).agg(
    n=('passes', 'count'),
    n_pass=('passes', 'sum'),
    mean_plddt=('cf_plddt', 'mean'),
).reset_index()
summary['pass_rate'] = summary['n_pass'] / summary['n']
print(summary.to_string(index=False))

# Save passing FASTA
MPNN_PASS_FASTA = CONTENT / 'mpnn_colabfold_passing.fasta'
passing = meta_df[meta_df['passes']]
with open(MPNN_PASS_FASTA, 'w') as f:
    for _, row in passing.iterrows():
        f.write(f">{row['bar_id']}_t{row['temp']}_d{row['design_idx']:02d} | {row['header']}\n")
        f.write(f"{row['sequence']}\n")
print(f'\nPassing sequences saved → {MPNN_PASS_FASTA}')

In [ ]:
# A6 — Figures
bar_ids  = sorted(meta_df['bar_id'].unique())
temps    = sorted(meta_df['temp'].unique())
temp_colors = {0.1: '#00d4ff', 0.2: '#ff9500', 0.3: '#cc44ff'}

# ── Fig 1: Pass rate by bar × temp (grouped bar chart) ──
fig, ax = plt.subplots(figsize=(14, 5), facecolor=DARK_BG)
dark_ax(ax)
x      = np.arange(len(bar_ids))
width  = 0.25
for j, temp in enumerate(temps):
    sub  = summary[summary['temp'] == temp].set_index('bar_id').reindex(bar_ids)
    vals = sub['pass_rate'].fillna(0).values
    ax.bar(x + (j - 1) * width, vals, width, label=f'temp={temp}',
           color=temp_colors[temp], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(bar_ids, rotation=45, ha='right', color=WHITE, fontsize=8)
ax.set_ylabel('Pass rate (pLDDT ≥ 0.35)', color=WHITE)
ax.set_ylim(0, 1.05)
ax.set_title('ProteinMPNN Pass Rate by Bar × Temperature (ColabFold)', color=WHITE)
ax.legend(facecolor='#1a1a1a', labelcolor=WHITE, fontsize=9)
plt.tight_layout()
FIG_A1 = CONTENT / 'fig_mpnn_pass_rate_by_temp.png'
plt.savefig(FIG_A1, dpi=150, facecolor=DARK_BG); plt.show()

# ── Fig 2: pLDDT violin per bar, coloured by temp ──
fig, axes = plt.subplots(1, len(bar_ids), figsize=(2.2 * len(bar_ids), 5), facecolor=DARK_BG)
if len(bar_ids) == 1: axes = [axes]
for ax, bar_id in zip(axes, bar_ids):
    dark_ax(ax)
    parts_all = []
    for j, temp in enumerate(temps):
        vals = meta_df[(meta_df['bar_id'] == bar_id) & (meta_df['temp'] == temp)]['cf_plddt'].dropna()
        if len(vals) < 2: continue
        vp = ax.violinplot(vals, positions=[j], widths=0.6, showmedians=True)
        for part in vp['bodies']:
            part.set_facecolor(temp_colors[temp])
            part.set_alpha(0.7)
        vp['cmedians'].set_color(WHITE)
    ax.axhline(PLDDT_MIN, color='#ff4444', lw=0.8, linestyle='--')
    ax.set_xticks(range(len(temps)))
    ax.set_xticklabels([str(t) for t in temps], fontsize=7, color=WHITE)
    ax.set_ylim(0, 1)
    ax.set_title(bar_id, fontsize=8, color=WHITE)
    if bar_id == bar_ids[0]: ax.set_ylabel('ColabFold pLDDT', color=WHITE)
plt.suptitle('pLDDT Distribution per Bar × Temperature', color=WHITE, fontsize=11, y=1.01)
plt.tight_layout()
FIG_A2 = CONTENT / 'fig_mpnn_plddt_violin.png'
plt.savefig(FIG_A2, dpi=150, facecolor=DARK_BG, bbox_inches='tight'); plt.show()

# ── Fig 3: Dropout recovery — bar_0, bar_8, bar_46 specifically ──
dropout_bars = ['bar_0', 'bar_8', 'bar_46']
dropout_df   = meta_df[meta_df['bar_id'].isin(dropout_bars)]
if len(dropout_df):
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=DARK_BG)
    dark_ax(ax)
    for bar_id in dropout_bars:
        for j, temp in enumerate(temps):
            vals = dropout_df[(dropout_df['bar_id'] == bar_id) & (dropout_df['temp'] == temp)]['cf_plddt'].dropna()
            ax.scatter([f'{bar_id}\nt={temp}'] * len(vals), vals,
                       color=temp_colors[temp], s=15, alpha=0.5)
    ax.axhline(PLDDT_MIN, color='#ff4444', lw=1, linestyle='--', label=f'threshold {PLDDT_MIN}')
    ax.set_ylabel('ColabFold pLDDT', color=WHITE)
    ax.set_title('Dropout Bar Recovery Across Temperatures', color=WHITE)
    ax.legend(facecolor='#1a1a1a', labelcolor=WHITE)
    plt.xticks(rotation=30, ha='right', color=WHITE)
    plt.tight_layout()
    FIG_A3 = CONTENT / 'fig_dropout_recovery.png'
    plt.savefig(FIG_A3, dpi=150, facecolor=DARK_BG); plt.show()
else:
    FIG_A3 = None
    print('No dropout bars in dataset.')

print('Figures saved.')

In [ ]:
# A7 — Download
zip_out = '/content/mpnn_results.zip'
with zipfile.ZipFile(zip_out, 'w') as zf:
    zf.write(MPNN_CSV,        'mpnn_colabfold_results.csv')
    zf.write(MPNN_PASS_FASTA, 'mpnn_colabfold_passing.fasta')
    zf.write(FIG_A1,          'fig_mpnn_pass_rate_by_temp.png')
    zf.write(FIG_A2,          'fig_mpnn_plddt_violin.png')
    if FIG_A3: zf.write(FIG_A3, 'fig_dropout_recovery.png')
files.download(zip_out)
print('Downloaded mpnn_results.zip')

---
---
# SECTION B — Originals + Scrambled → ColabFold (Apples-to-Apples)

**Run this in a fresh session** (or after Section A finishes) to avoid GPU memory conflicts.

All four sequence buckets are folded with the **same ColabFold model** so pLDDT values are directly comparable:
- `concordance` — original lyric-derived sequence
- `native_ala` — alanine substitution at BJOZXU positions
- `scrambled_concordance` — concordance sequence shuffled (same AA composition, random order)
- `scrambled_native_ala` — native_ala shuffled

Expected: scrambled ≈ originals under ESMFold (ESMFold is insensitive to order at this length).
ColabFold uses AlphaFold2 + MSA — order-sensitive, more likely to discriminate.

In [ ]:
# B1 — Load sequences
enriched = pd.read_csv(DATA / 'aggregated_lines_v2_enriched.csv')
cands    = pd.read_csv(DATA / 'phase2_candidates.csv')
enriched = enriched[enriched['bar_id'].isin(set(cands['bar_id']))]
enriched = enriched.dropna(subset=['fasta_seq_concordance']).reset_index(drop=True)
print(f'{len(enriched)} bars')

rng  = random.Random(SEED)
rows = []

for _, row in enriched.iterrows():
    bar_id   = row['bar_id']
    conc_seq = str(row.get('fasta_seq_concordance', '') or '')
    na_seq   = str(row.get('fasta_seq_native_alanine', '') or '')

    # Originals
    if conc_seq:
        rows.append({'bar_id': bar_id, 'source': 'concordance',
                     'shuffle_idx': -1, 'sequence': conc_seq, 'seq_len': len(conc_seq)})
    if na_seq:
        rows.append({'bar_id': bar_id, 'source': 'native_ala',
                     'shuffle_idx': -1, 'sequence': na_seq, 'seq_len': len(na_seq)})
    # Scrambles
    for i in range(N_SHUFFLES):
        if conc_seq:
            rows.append({'bar_id': bar_id, 'source': 'scrambled_concordance',
                         'shuffle_idx': i, 'sequence': shuffle_sequence(conc_seq, rng), 'seq_len': len(conc_seq)})
        if na_seq:
            rows.append({'bar_id': bar_id, 'source': 'scrambled_native_ala',
                         'shuffle_idx': i, 'sequence': shuffle_sequence(na_seq, rng), 'seq_len': len(na_seq)})

sc_df = pd.DataFrame(rows)
sc_df['cf_name'] = (
    sc_df['bar_id'] + '_' +
    sc_df['source'].str.replace('scrambled_', 'sc').str.replace('concordance', 'conc').str.replace('native_ala', 'na') +
    '_' + sc_df['shuffle_idx'].astype(str)
)
print(f'Total sequences: {len(sc_df)}')
print(sc_df.groupby('source').size())

In [ ]:
# B2 — Write single FASTA (originals + scrambles together)
CF_SC_FASTA = CONTENT / 'originals_and_scrambled.fasta'
with open(CF_SC_FASTA, 'w') as f:
    for _, row in sc_df.iterrows():
        f.write(f">{row['cf_name']}\n{row['sequence']}\n")
print(f'Wrote {len(sc_df)} sequences → {CF_SC_FASTA}')

In [ ]:
# B3 — Run ColabFold (all 4 buckets in one batch)
# ~12 bars × (2 orig + 6 scrambles) = ~96 seqs on A100 ≈ 5 min
CF_SC_OUT  = CONTENT / 'cf_sc_output'
sc_scores  = run_colabfold(CF_SC_FASTA, CF_SC_OUT)
print(f'Scored {len(sc_scores)} sequences.')

In [ ]:
# B4 — Merge scores
sc_df['cf_plddt'] = sc_df['cf_name'].map(lambda n: sc_scores.get(n, {}).get('plddt'))
sc_df['cf_ptm']   = sc_df['cf_name'].map(lambda n: sc_scores.get(n, {}).get('ptm'))

SC_CSV = CONTENT / 'scrambled_colabfold_results.csv'
sc_df.to_csv(SC_CSV, index=False)
print(f'Saved {len(sc_df)} rows → {SC_CSV}')

print('\n--- pLDDT by bucket ---')
for src in ['concordance', 'native_ala', 'scrambled_concordance', 'scrambled_native_ala']:
    vals = sc_df.loc[sc_df['source'] == src, 'cf_plddt'].dropna()
    if len(vals):
        print(f'  {src:25s}  n={len(vals):3d}  mean={vals.mean():.3f}  '
              f'sd={vals.std():.3f}  <0.3: {(vals < 0.3).sum()}/{len(vals)}')

In [ ]:
# B5 — Figures
COLORS = {
    'concordance':           '#00d4ff',
    'native_ala':            '#ff9500',
    'scrambled_concordance': '#444455',
    'scrambled_native_ala':  '#664444',
}
LABELS = {
    'concordance':           'concordance (ColabFold)',
    'native_ala':            'native_ala (ColabFold)',
    'scrambled_concordance': 'scrambled concordance',
    'scrambled_native_ala':  'scrambled native_ala',
}

bar_ids = list(sc_df['bar_id'].unique())
x_pos   = {b: i for i, b in enumerate(bar_ids)}

# ── Fig B1: per-bar scatter (4 sources) ──
fig, ax = plt.subplots(figsize=(max(10, len(bar_ids)*0.9), 5), facecolor=DARK_BG)
dark_ax(ax)
plotted = set()
for _, row in sc_df.iterrows():
    plddt = row['cf_plddt']
    if plddt is None or pd.isna(plddt): continue
    src     = row['source']
    x       = x_pos[row['bar_id']]
    is_orig = row['shuffle_idx'] == -1
    label   = LABELS.get(src, src) if src not in plotted else ''
    ax.scatter(x, plddt, color=COLORS.get(src,'#888'),
               s=70 if is_orig else 20,
               marker='D' if is_orig else 'o',
               alpha=0.85, zorder=4 if is_orig else 2, label=label)
    plotted.add(src)
ax.axhline(0.3, color='#ff4444', lw=1, linestyle='--', label='pLDDT=0.3')
ax.set_xticks(range(len(bar_ids)))
ax.set_xticklabels(bar_ids, rotation=45, ha='right', fontsize=7, color=WHITE)
ax.set_ylabel('ColabFold pLDDT', color=WHITE)
ax.set_ylim(0, 1)
ax.set_title('Concordance vs Native_Ala vs Scrambled — ColabFold (Apples-to-Apples)', color=WHITE)
handles, labels_ = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels_):
    if l and l not in seen: seen[l] = h
ax.legend(seen.values(), seen.keys(), facecolor='#1a1a1a', labelcolor=WHITE, fontsize=8)
plt.tight_layout()
FIG_B1 = CONTENT / 'fig_scrambled_cf_scatter.png'
plt.savefig(FIG_B1, dpi=150, facecolor=DARK_BG); plt.show()

# ── Fig B2: box plot — overall distribution by bucket ──
fig, ax = plt.subplots(figsize=(8, 5), facecolor=DARK_BG)
dark_ax(ax)
sources_ord = ['concordance', 'native_ala', 'scrambled_concordance', 'scrambled_native_ala']
data_by_src = [sc_df.loc[sc_df['source']==s, 'cf_plddt'].dropna().values for s in sources_ord]
bp = ax.boxplot(data_by_src, patch_artist=True, medianprops={'color': WHITE, 'lw': 2})
for patch, src in zip(bp['boxes'], sources_ord):
    patch.set_facecolor(COLORS[src])
    patch.set_alpha(0.8)
for elem in ['whiskers', 'caps', 'fliers']:
    for part in bp[elem]: part.set_color('#888')
ax.axhline(0.3, color='#ff4444', lw=1, linestyle='--', label='pLDDT=0.3')
ax.set_xticks(range(1, len(sources_ord)+1))
ax.set_xticklabels([LABELS[s] for s in sources_ord], rotation=20, ha='right', color=WHITE, fontsize=8)
ax.set_ylabel('ColabFold pLDDT', color=WHITE)
ax.set_ylim(0, 1)
ax.set_title('pLDDT Distribution: Originals vs Scrambled (ColabFold)', color=WHITE)
ax.legend(facecolor='#1a1a1a', labelcolor=WHITE)
plt.tight_layout()
FIG_B2 = CONTENT / 'fig_scrambled_cf_boxplot.png'
plt.savefig(FIG_B2, dpi=150, facecolor=DARK_BG); plt.show()

# ── Fig B3: delta pLDDT — original minus scrambled mean per bar ──
delta_rows = []
for bar_id in bar_ids:
    sub = sc_df[sc_df['bar_id'] == bar_id]
    for src, sc_src in [('concordance', 'scrambled_concordance'), ('native_ala', 'scrambled_native_ala')]:
        orig_val = sub.loc[sub['source']==src, 'cf_plddt'].mean()
        sc_val   = sub.loc[sub['source']==sc_src, 'cf_plddt'].mean()
        if pd.notna(orig_val) and pd.notna(sc_val):
            delta_rows.append({'bar_id': bar_id, 'source': src, 'delta': orig_val - sc_val})
delta_df = pd.DataFrame(delta_rows)

fig, ax = plt.subplots(figsize=(max(10, len(bar_ids)*0.9), 4), facecolor=DARK_BG)
dark_ax(ax)
x = np.arange(len(bar_ids))
for j, src in enumerate(['concordance', 'native_ala']):
    sub = delta_df[delta_df['source']==src].set_index('bar_id').reindex(bar_ids)['delta']
    ax.bar(x + (j-0.5)*0.3, sub.values, 0.3, color=COLORS[src], alpha=0.85, label=src)
ax.axhline(0, color='#888', lw=0.8)
ax.axhline(0, color='#ff4444', lw=0, linestyle='--')  # placeholder
ax.set_xticks(x)
ax.set_xticklabels(bar_ids, rotation=45, ha='right', fontsize=7, color=WHITE)
ax.set_ylabel('pLDDT(original) − pLDDT(scrambled)', color=WHITE)
ax.set_title('Structure Signal: Original − Scrambled pLDDT per Bar', color=WHITE)
ax.legend(facecolor='#1a1a1a', labelcolor=WHITE, fontsize=8)
plt.tight_layout()
FIG_B3 = CONTENT / 'fig_scrambled_delta_plddt.png'
plt.savefig(FIG_B3, dpi=150, facecolor=DARK_BG); plt.show()
print('Positive delta = original folds better than scramble (structure is sequence-order dependent).')
print('Zero or negative delta = ESMFold/ColabFold cannot discriminate at this length.')

In [ ]:
# B6 — Download
zip_out = '/content/scrambled_results.zip'
with zipfile.ZipFile(zip_out, 'w') as zf:
    zf.write(SC_CSV,  'scrambled_colabfold_results.csv')
    zf.write(FIG_B1,  'fig_scrambled_cf_scatter.png')
    zf.write(FIG_B2,  'fig_scrambled_cf_boxplot.png')
    zf.write(FIG_B3,  'fig_scrambled_delta_plddt.png')
files.download(zip_out)
print('Downloaded scrambled_results.zip')